In [1]:
# Import Libraries
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from services import league_services
from db.conn import get_connection
from db.repo import MatchRepo, TimelineRepo

In [2]:
# Get Summoner Information
server = "na1"
summoner_name = "STEAL#3v3"
summoner = league_services.register_summoner(server, summoner_name=summoner_name)
summoner_db = league_services.get_stored_summoner(server, summoner_name)
puuid = summoner_db["puuid"]
# print(summoner_db)

In [3]:
# Get Match Data From DB
# Participants
with get_connection() as conn:
    participants = MatchRepo(conn).get_participants_by_puuid(puuid)
participants_df = pd.DataFrame(participants)

# Matches, Teams, Subteams, All Participants, Timelines
match_ids = participants_df["match_id"].unique().tolist()
with get_connection() as conn:
    matches = MatchRepo(conn).get_matches(match_ids)
    teams = MatchRepo(conn).get_teams(match_ids)
    subteams = MatchRepo(conn).get_subteams(match_ids)
    all_participants = MatchRepo(conn).get_all_participants(match_ids)
    match_timelines = TimelineRepo(conn).get_all_timelines(match_ids)
    timeline_frames = TimelineRepo(conn).get_all_timeline_frames(match_ids)
    participant_frames = TimelineRepo(conn).get_all_participant_frames(match_ids)
    timeline_events = TimelineRepo(conn).get_all_timeline_events(match_ids)

print(matches[0].keys())
print(teams[0].keys())
print(subteams[0].keys())
print(participants[0].keys())
print(all_participants[0].keys())
print(match_timelines[0].keys())
print(timeline_frames[0].keys())
print(participant_frames[0].keys())
print(timeline_events[0].keys())

dict_keys(['match_id', 'platform_id', 'game_id', 'end_of_game_result', 'data_version', 'game_creation', 'game_duration', 'game_end_timestamp', 'game_start_timestamp', 'map_id', 'queue_id', 'game_mode', 'game_type', 'game_version', 'patch'])
dict_keys(['match_id', 'team_id', 'win', 'bans', 'champion_kills', 'first_blood', 'baron_kills', 'baron_first', 'dragon_kills', 'dragon_first', 'inhibitor_kills', 'inhibitor_first', 'rift_herald_kills', 'rift_herald_first', 'tower_kills', 'tower_first'])
dict_keys(['match_id', 'subteam_id', 'placement'])
dict_keys(['match_id', 'participant_id', 'team_id', 'puuid', 'riot_id_game_name', 'riot_id_tagline', 'summoner_level', 'win', 'champion_id', 'champion_name', 'champion_level', 'champion_experience', 'team_position', 'individual_position', 'lane', 'player_role', 'kills', 'deaths', 'assists', 'total_minions_killed', 'neutral_minions_killed', 'total_ally_jungle_minions_killed', 'total_enemy_jungle_minions_killed', 'killing_sprees', 'largest_killing_spr

In [4]:
# 0.1 Available datasets
matches_df = pd.DataFrame(matches)
teams_df = pd.DataFrame(teams)
subteams_df = pd.DataFrame(subteams)
participants_df = pd.DataFrame(participants)
all_participants_df = pd.DataFrame(all_participants)

match_timelines_df = pd.DataFrame(match_timelines)
timeline_frames_df = pd.DataFrame(timeline_frames)
participant_frames_df = pd.DataFrame(participant_frames)
timeline_events_df = pd.DataFrame(timeline_events)

In [5]:
# 0.2 Filters
matches_df["is_remake"] = matches_df["game_duration"] < 300
valid_matches_df = matches_df[matches_df["is_remake"] == False].copy()
valid_match_ids = valid_matches_df["match_id"].unique()
valid_participants_df = participants_df[participants_df["match_id"].isin(valid_match_ids)].copy()
valid_participants_df["team_position"] = (
    valid_participants_df["team_position"]
    .replace("", "NO ROLE")
)

In [6]:
# 1. OVERALL PERFORMANCE DASHBOARD
# 1.1 Career / database overview


In [9]:
# 1.1.1 Core Volume
total_matches = len(valid_matches_df)
total_wins = valid_participants_df["win"].sum()
total_losses = (valid_participants_df["win"] == 0).sum()
win_rate = (total_wins / total_matches) * 100

total_playtime = valid_matches_df["game_duration"].sum()
average_duration = valid_matches_df["game_duration"].mean()
total_hours = total_playtime / 3600
average_minutes = average_duration / 60

valid_matches_df["game_date"] = pd.to_datetime(
    valid_matches_df["game_creation"],
    unit="ms"
)
games_per_day = (
    valid_matches_df
    .groupby(valid_matches_df["game_date"].dt.date)
    .size()
)
games_per_week = (
    valid_matches_df
    .groupby(valid_matches_df["game_date"].dt.to_period("W"))
    .size()
)
games_per_month = (
    valid_matches_df
    .groupby(valid_matches_df["game_date"].dt.to_period("M"))
    .size()
)

unique_champions = valid_participants_df["champion_name"].nunique()
champions_played = valid_participants_df["champion_name"].nunique()
champion_list = valid_participants_df["champion_name"].unique()

unique_roles = valid_participants_df["team_position"].nunique()
roles_played = (
    valid_participants_df["team_position"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

unique_patches = valid_matches_df["patch"].nunique()
patches_played = valid_matches_df["patch"].unique()

unique_queues = valid_matches_df["queue_id"].nunique()
unique_game_modes = valid_matches_df["game_mode"].nunique()

def format_duration(seconds):
    hours, remainder = divmod(int(seconds), 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours}h {minutes}m {seconds}s"

core_volume = {
    "total_matches": total_matches,

    "total_wins": total_wins,
    "total_losses": total_losses,
    "win_rate": round(win_rate, 2),

    "total_playtime": format_duration(total_playtime),
    "total_playtime_hours": round(total_hours, 2),
    "average_match_duration": format_duration(average_duration),

    "unique_champions": unique_champions,
    "unique_roles": unique_roles,

    "unique_patches": unique_patches,
    "unique_queues": unique_queues,
    "unique_game_modes": unique_game_modes,
}
print(core_volume)

{'total_matches': 660, 'total_wins': np.int64(426), 'total_losses': np.int64(234), 'win_rate': np.float64(64.55), 'total_playtime': '245h 49m 17s', 'total_playtime_hours': np.float64(245.82), 'average_match_duration': '0h 22m 20s', 'unique_champions': 115, 'unique_roles': 6, 'unique_patches': 20, 'unique_queues': 11, 'unique_game_modes': 4}
